In [0]:
from pyspark.sql.functions import col

# Read Gold transaction data
df = spark.table("workspace.default.fact_transactions")

print("Total rows:", df.count())

print("Normal transactions:")
print(df.filter(col("Class") == 0).count())

print("Fraud transactions:")
print(df.filter(col("Class") == 1).count())

Total rows: 283726
Normal transactions:
283253
Fraud transactions:
473


In [0]:
from pyspark.sql.functions import col

# Read Gold data
df = spark.table("workspace.default.fact_transactions")

# Remove ingestion metadata from ML features
ml_df = df.drop("_ingestion_time")

# Train / Test split
train_df, test_df = ml_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

print("Training fraud:", train_df.filter(col("Class") == 1).count())
print("Testing fraud:", test_df.filter(col("Class") == 1).count())

Training rows: 227179
Testing rows: 56547
Training fraud: 387
Testing fraud: 86


In [0]:
from pyspark.ml.feature import VectorAssembler

# ML feature columns
feature_columns = [
    "V1", "V2", "V3", "V4", "V5", "V6", "V7",
    "V8", "V9", "V10", "V11", "V12", "V13", "V14",
    "V15", "V16", "V17", "V18", "V19", "V20", "V21",
    "V22", "V23", "V24", "V25", "V26", "V27", "V28",
    "Amount"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

train_features = assembler.transform(train_df)
test_features = assembler.transform(test_df)

train_features.select("features", "Class").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [0]:
from pyspark.sql.functions import col, when
from pyspark.ml.classification import LogisticRegression

# Create class weights
fraud_count = train_features.filter(col("Class") == 1).count()
normal_count = train_features.filter(col("Class") == 0).count()

total_count = fraud_count + normal_count

train_weighted = train_features.withColumn(
    "class_weight",
    when(
        col("Class") == 1,
        total_count / (2.0 * fraud_count)
    ).otherwise(
        total_count / (2.0 * normal_count)
    )
)

# Create Logistic Regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="Class",
    weightCol="class_weight",
    maxIter=50
)

# Train model
model = lr.fit(train_weighted)

print("Model training completed!")

Model training completed!


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

# Generate predictions on unseen test data
predictions = model.transform(test_features)

# Show sample predictions
predictions.select(
    "Class",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+-----------------------------------------+
|Class|prediction|probability                              |
+-----+----------+-----------------------------------------+
|0    |0.0       |[0.9612723586657039,0.03872764133429607] |
|0    |0.0       |[0.9735695292304587,0.026430470769541303]|
|0    |0.0       |[0.823837459193178,0.17616254080682203]  |
|0    |0.0       |[0.8929656502276764,0.10703434977232362] |
|0    |0.0       |[0.802065387394289,0.197934612605711]    |
|0    |0.0       |[0.9367212246639011,0.06327877533609894] |
|0    |0.0       |[0.9967636479317382,0.003236352068261783]|
|0    |0.0       |[0.9340173416691908,0.06598265833080919] |
|0    |0.0       |[0.9679062631898238,0.03209373681017624] |
|0    |0.0       |[0.9901706164254866,0.00982938357451335] |
+-----+----------+-----------------------------------------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Confusion matrix counts
tp = predictions.filter(
    (col("Class") == 1) & (col("prediction") == 1)
).count()

tn = predictions.filter(
    (col("Class") == 0) & (col("prediction") == 0)
).count()

fp = predictions.filter(
    (col("Class") == 0) & (col("prediction") == 1)
).count()

fn = predictions.filter(
    (col("Class") == 1) & (col("prediction") == 0)
).count()


# Metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)


print("Confusion Matrix")
print("----------------")
print("True Positives :", tp)
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)

print("\nMetrics")
print("----------------")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Confusion Matrix
----------------
True Positives : 82
True Negatives : 55123
False Positives: 1338
False Negatives: 4

Metrics
----------------
Precision: 0.0577
Recall   : 0.9535
F1 Score : 0.1089


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="Class",
    rawPredictionCol="probability",
    metricName="areaUnderPR"
)

pr_auc = evaluator.evaluate(predictions)

print(f"PR-AUC: {pr_auc:.4f}")

PR-AUC: 0.6808


In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    lit,
    monotonically_increasing_id
)

from pyspark.ml.functions import vector_to_array


# Convert ML probability vector to array
fraud_predictions = (
    predictions
    .withColumn(
        "fraud_probability",
        vector_to_array(col("probability"))[1]
    )
    .withColumn(
        "prediction_time",
        current_timestamp()
    )
    .withColumn(
        "model_version",
        lit("logistic_regression_v1")
    )
    .withColumn(
        "prediction_id",
        monotonically_increasing_id()
    )
)


# Select relevant columns
fraud_predictions_gold = fraud_predictions.select(
    "prediction_id",
    "Time",
    "Amount",
    "Class",
    "prediction",
    "fraud_probability",
    "model_version",
    "prediction_time"
)


# Save Gold prediction table
(
    fraud_predictions_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.fact_fraud_predictions"
    )
)

print("Fraud prediction Gold table created!")

Fraud prediction Gold table created!


In [0]:
spark.table(
    "workspace.default.fact_fraud_predictions"
).display()

prediction_id,Time,Amount,Class,prediction,fraud_probability,model_version,prediction_time
0,7.0,40.8,0,0.0,0.03872764133429607,logistic_regression_v1,2026-09-26T09:05:32.351Z
1,10.0,121.5,0,0.0,0.026430470769541303,logistic_regression_v1,2026-09-26T09:05:32.351Z
2,11.0,27.5,0,0.0,0.17616254080682203,logistic_regression_v1,2026-09-26T09:05:32.351Z
3,17.0,34.09,0,0.0,0.10703434977232362,logistic_regression_v1,2026-09-26T09:05:32.351Z
4,18.0,2.28,0,0.0,0.197934612605711,logistic_regression_v1,2026-09-26T09:05:32.351Z
5,22.0,26.43,0,0.0,0.06327877533609894,logistic_regression_v1,2026-09-26T09:05:32.351Z
6,22.0,0.89,0,0.0,0.003236352068261783,logistic_regression_v1,2026-09-26T09:05:32.351Z
7,23.0,12.99,0,0.0,0.06598265833080919,logistic_regression_v1,2026-09-26T09:05:32.351Z
8,24.0,17.28,0,0.0,0.03209373681017624,logistic_regression_v1,2026-09-26T09:05:32.351Z
9,27.0,1.8,0,0.0,0.00982938357451335,logistic_regression_v1,2026-09-26T09:05:32.351Z


In [0]:
print(
    "Prediction rows:",
    spark.table(
        "workspace.default.fact_fraud_predictions"
    ).count()
)

Prediction rows: 56547
